In [31]:
from dotenv import load_dotenv
load_dotenv()

# Create and API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-6"

import json

def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})

def add_assistant_message(messages, content):
    messages.append({"role": "assistant", "content": content})

# This function uses output_config to specify that the output should be in JSON format, with a specific schema. 
# The schema defines an array of objects, where each object has a "task" property of type string. 
# This allows for structured output from the model, which can be useful for downstream processing or validation.
def chat(messages, system_prompt=None, temperature=1.0, output_config=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    
    if system_prompt:
        params["system"] = system_prompt

    if output_config:
        params["output_config"] = output_config
    
    response = client.messages.create(**params)

    return response.content[0].text

In [40]:
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete, and the format that for used.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    json_output_config = {
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "task": {"type": "string"},
                        "format": {"type": "string", "enum": ["python", "json", "regex"]},
                    },
                    "required": ["task"],
                    "additionalProperties": False
                }
            }
        }
    }

    messages = []
    add_user_message(messages, prompt)
    chat_response = chat(messages, output_config=json_output_config)

    return json.loads(chat_response)  # This will parse the JSON response into a Python object

In [50]:
def run_prompt(test_case):
    """Merges the prompt with the test case and returns the result"""
    prompt = f"""
Please solve the following task:

{test_case['task']}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [34]:
def grade_by_model(test_case, output):
    """Validate that the test_case output clearly solves the task, and that generated code is accurate and correct."""

    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case["task"]}
    Solution: {output}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """

    messages = []
    add_user_message(messages, eval_prompt)

    json_output_config = {
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "strengths": {
                        "type": "array",
                        "items": {"type": "string"}
                    },
                    "weaknesses": {
                        "type": "array",
                        "items": {"type": "string"}
                    },
                    "reasoning": {"type": "string"},
                    "score": {"type": "number"}
                },
                "required": ["strengths", "weaknesses", "reasoning", "score"],
                "additionalProperties": False
            }
        }
    }
    
    eval_text = chat(messages, output_config=json_output_config)
    return json.loads(eval_text)

In [48]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(response, test_case):
    """Validate the syntax of the response based on the task type"""
    format = test_case["format"].lower()
    if format == "python":
        return validate_python(response)
    elif format == "json":
        return validate_json(response)
    elif format == "regex":
        return validate_regex(response)

    return None

In [49]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result as a dictionary"""
    output = run_prompt(test_case)

    # Grading: 10 is great output. 1 is terrible output.
    # Evaluation criteria: 
    # * Only valid Python, JSON, or Regex output is acceptable.      <-- Validated using code
    # * Valid syntax                                                 <-- Validated using code
    # * Respose shall clearly solve the task. Generated code         <-- Validated using Claude
    #   shall be accurate and correct.
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [37]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case for each test case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean(result["score"] for result in results)
    print(f"Average score: {average_score}")

    return results

In [38]:
with open("005_test_dataset.json", "r") as f:
    dataset = json.load(f)

result = run_eval(dataset)

print(json.dumps(result, indent=2))

Average score: 6.666666666666667
[
  {
    "output": "# AWS S3 Object Key Listing Function\n\n## Implementation\n\n```python\nimport boto3\nfrom botocore.exceptions import ClientError, NoCredentialsError\nfrom typing import Optional\n\n\ndef list_s3_objects(\n    bucket_name: str,\n    prefix: str = \"\",\n    region_name: Optional[str] = None,\n    aws_access_key_id: Optional[str] = None,\n    aws_secret_access_key: Optional[str] = None,\n) -> list[str]:\n    \"\"\"\n    List all object keys in an S3 bucket matching a given prefix.\n\n    Args:\n        bucket_name: The name of the S3 bucket.\n        prefix: The prefix string to filter objects (default: \"\" lists all objects).\n        region_name: AWS region name (optional, uses default if not provided).\n        aws_access_key_id: AWS access key ID (optional, uses default credentials if not provided).\n        aws_secret_access_key: AWS secret access key (optional, uses default credentials if not provided).\n\n    Returns:\n      

In [41]:
response = generate_dataset()
response
import json
with open("005_test_dataset.json", "w") as f:
    json.dump(response, f)
